# Tables and documents in one graph

A walkthrough you can watch happen. Each step writes to a live FalkorDB graph and
gives you a query to paste into the browser, so you see the graph change rather
than take my word for it.

The scenario is ordinary. An energy company keeps two kinds of thing:

| | |
|---|---|
| **Documents** | a board review and a market note — prose, ingested the usual way |
| **Tables** | an org export, an employee export, a contracts export — CSVs, ingested by declaration |

Neither half can answer the interesting question alone. The board review knows
who had a shortfall but nothing about headcount. The employee export knows ages
and titles but has never heard of a shortfall.

**You need:** FalkorDB on `localhost:6379` with its browser on `:3000`, and
`OPENAI_API_KEY` in your environment.

```
docker run -p 6379:6379 -p 3000:3000 falkordb/falkordb
```

## 0. The files

Seven small files live next to this notebook in `data/hybrid_walkthrough/`. They
are synthetic but real files on disk — open them, edit them, point the notebook
at your own instead.

| | |
|---|---|
| `board_review.pdf`, `market_note.pdf` | the prose |
| `organizations.csv`, `employees.csv`, `contracts.csv` | the tables |
| `employees_v2.csv` | the same export a quarter later, for step 8 |
| `tickets.csv` | a table of free text, for step 9 |

In [ ]:
from pathlib import Path

DATA = Path("data/hybrid_walkthrough")
assert DATA.is_dir(), f"run this notebook from graphrag_sdk/examples/ (looked in {DATA.resolve()})"

for path in sorted(DATA.glob("*.[cp][sd][vf]")):
    print(f"  {path.name:22} {path.stat().st_size:>7,} bytes")

### The names are the join

Every name in the tables appears **identically** in the prose:
`Northwind Energy`, `Kestrel Grid`, `Maya Ellison`. That is not decoration — the
SDK joins a row to a mention by exact string equality on the name, so a table
whose display names are phrases the documents never use will not join to
anything. Choosing join-able names is the one design decision that matters most.

## 1. Connect

One graph for the whole walkthrough, reset so you can re-run the notebook from the top.

In [ ]:
import os
import textwrap

from graphrag_sdk import (
    Column, ConnectionConfig, GraphRAG, Link, LiteLLM, LiteLLMEmbedder, Table, TextLoader,
)

GRAPH = "hybrid_walkthrough"
MODEL = os.environ.get("GRAPHRAG_MODEL", "gpt-4o-mini")
EMBEDDER = os.environ.get("GRAPHRAG_EMBEDDER", "text-embedding-3-small")
KEY = os.environ["OPENAI_API_KEY"]          # raises here rather than mid-ingest

rag = GraphRAG(
    connection=ConnectionConfig(host="localhost", port=6379, graph_name=GRAPH),
    llm=LiteLLM(model=f"openai/{MODEL}", api_key=KEY, temperature=0.0),
    embedder=LiteLLMEmbedder(model=f"openai/{EMBEDDER}", api_key=KEY, dimensions=256),
    embedding_dimension=256,
    # What turns a question into a query against the declared column types.
    # Without it "what is the average age" has no passage to retrieve.
    enable_cypher=True,
)
await rag.__aenter__()
await rag.query("MATCH (n) DETACH DELETE n")
print(f"graph {GRAPH!r} is empty. Browser: http://localhost:3000")

### A helper for watching the graph

Every step prints a query to paste into the browser, and what you should expect
to see. `look()` also runs it here so the notebook stays self-contained.

In [ ]:
def _render(value):
    """Nodes come back as objects; show them as Label(name) so output is readable.

    The queries return whole nodes on purpose — that is what draws a picture in
    the browser — but printed here they would be <Node object at 0x...>.
    """
    labels = getattr(value, "labels", None)
    props = getattr(value, "properties", None)
    if labels is None or props is None:
        return value
    kind = next((l for l in labels if l != "__Entity__"), "Node")
    shown = props.get("name") or props.get("record_key") or props.get("id") or ""
    return f"{kind}({shown})" if shown else kind


async def look(what: str, cypher: str, expect: str = "") -> list:
    rows = await rag.query(cypher)
    print(f"\n{'-' * 78}\n{what}\n{'-' * 78}")
    print("paste into http://localhost:3000  (graph: " + GRAPH + ")\n")
    print(textwrap.indent(cypher.strip(), "    "))
    if expect:
        print(f"\n  expect: {expect}")
    print(f"\n  {len(rows)} row(s):")
    for row in rows[:12]:
        print("    ", [_render(value) for value in row])
    if len(rows) > 12:
        print(f"     ... and {len(rows) - 12} more")
    return rows

async def counts(label_note: str = "") -> None:
    rows = await rag.query(
        "MATCH (n) RETURN head([l IN labels(n) WHERE l <> '__Entity__']) AS label, "
        "count(n) AS n ORDER BY n DESC")
    total = sum(r[1] for r in rows)
    print(f"  {label_note}{'' if not label_note else '  '}{total} nodes: "
          + ", ".join(f"{r[0]}={r[1]}" for r in rows if r[0]))

## 2. The documents first

The ordinary path: load, chunk, then a model reads each chunk and proposes
entities. Nothing here knows anything about tables.

In [ ]:
for name in ("board_review.pdf", "market_note.pdf"):
    result = await rag.ingest(str(DATA / name))
    print(f"  {name:20} {result.nodes_created:>3} entities, {result.chunks_indexed:>2} chunks")

await counts("after the PDFs:")

In [ ]:
await look(
    "The lexical graph: a Document, its Chunks, and what was extracted from them",
    """MATCH (d:Document)-[:PART_OF]->(c:Chunk)<-[:MENTIONED_IN]-(e:__Entity__)
RETURN d, c, e
LIMIT 40""",
    "two documents, their chunks, and a cloud of extracted entities around them",
)

await look(
    "What the model decided each thing was",
    """MATCH (e:__Entity__)
RETURN head([l IN labels(e) WHERE l <> '__Entity__']) AS label,
       collect(e.name)[0..6] AS examples, count(e) AS n
ORDER BY n DESC""",
    "labels chosen from the ontology's menu — Organization, Person, Concept, and so on",
)

## 3. Now a table — and let the SDK write the mapping

A mapping says how a row becomes an entity. You can write it by hand, but
`propose_mapping` fits one to the ontology that already exists, which matters:
the extractor has just filed these companies under some label, and a mapping that
invents a different one for the same things would hold each company twice.

Notice how little the model is asked. The key column, every column's type, and
which columns are foreign keys are **measured** from the data. Only the label and
the relationship names are judgement.

In [ ]:
proposal = await rag.propose_mapping(str(DATA / "organizations.csv"))
print(proposal.summary())
print("\n" + "=" * 78 + "\ncommittable code:\n")
print(proposal.as_code())
print(f"\nintroduces a type the ontology lacked? {proposal.introduces_a_new_type}")

### Run that cell twice and you may get a different answer

The measured half will not move: the key, the types and the detected foreign keys
are read from the data. The judgement half can. On one run the model made
`country` a plain property; on another it linked it to `Location`, so `Norway`
became an entity joined to the market note. Both are defensible.

That is the argument for `as_code()`. Commit one answer and the mapping stops
being a model output and becomes part of your code, which is also what keeps a
load deterministic — a proposal regenerated per run is a model in the ingest path.

Accept it and load the table. Each row becomes a Chunk as well as an entity, so a row is traceable and retrievable exactly as a paragraph is.

In [ ]:
ORGS = proposal.table
result = await rag.ingest(str(DATA / "organizations.csv"), mapping=ORGS)
print(f"  {result}")
await counts("after organizations.csv:")

await look(
    "A row, as it lands: a typed entity plus the Chunk it came from",
    """MATCH (o:Organization)-[:MENTIONED_IN]->(c:Chunk {kind: 'record'})
RETURN o.name, o.country, o.employee_count, o.revenue_musd, c.record_key, c.text""",
    "employee_count is a number, not the text '1240' — that is the whole point of declaring types",
)

## 4. A table with a foreign key

`org_id` in the employee export is not text, it is an edge. `Link` says so. The
target is written ON CREATE only, so a pointer can never overwrite the name
`organizations.csv` supplied — which is why the two files can arrive in either
order.

In [ ]:
EMPLOYEES = Table(
    "Person",
    key="employee_id",
    name="full_name",
    age=Column("age", "INTEGER"),
    title=Column("job_title"),
    start_date=Column("start_date", "DATE"),
    links=[Link("WORKS_AT", to="Organization", by="org_id")],
)

EMPLOYEE_EXPORT = str(DATA / "employees.csv")   # the export's identity
result = await rag.ingest(EMPLOYEE_EXPORT, mapping=EMPLOYEES,
                          document_id=EMPLOYEE_EXPORT)
print(f"  {result}")

await look(
    "The employment edges the CSV created",
    """MATCH (p:Person)-[r:RELATES]->(o:Organization)
WHERE r.rel_type = 'WORKS_AT'
RETURN p, r, o""",
    "four edges from the CSV, plus the two the board review already asserted "
    "in prose — those merge in step 6. Every data edge is RELATES, with the "
    "meaning in rel_type",
)

## 5. Two links from one row

`Contract` is genuinely a new type, so ingesting this table logs a warning
listing the labels that already hold entities. That is the guard working, not a
problem: it exists because declaring `Person` beside an `Employee` that already
has data would hold each person twice, and it cannot tell that case from this one
without you.

A contract has a buyer and a seller, both organizations. Two links to the same
label from one row, which is the case that needs distinct handles internally —
you just write it.

In [ ]:
CONTRACTS = Table(
    "Contract",
    key="contract_id",
    name="contract_name",
    value_musd=Column("value_musd", "FLOAT"),
    signed_date=Column("signed_date", "DATE"),
    links=[
        Link("BUYER", to="Organization", by="buyer_org_id"),
        Link("SELLER", to="Organization", by="seller_org_id"),
    ],
)

result = await rag.ingest(str(DATA / "contracts.csv"), mapping=CONTRACTS)
print(f"  {result}")

await look(
    "Both sides of each contract",
    """MATCH (buyer:Organization)<-[b:RELATES]-(k:Contract)-[s:RELATES]->(seller:Organization)
WHERE b.rel_type = 'BUYER' AND s.rel_type = 'SELLER'
RETURN buyer.name, k.name, k.value_musd, seller.name""",
    "two contracts, each pointing at two different organizations",
)

## 6. finalize() — where the halves join

Until now these are two graphs sharing a database. The documents talk about
`Northwind Energy`; the table has a row keyed `ORG-NW` whose name is also
`Northwind Energy`. Resolution matches on name **and** label, so they merge, and
the keyed id survives because it is the one the next load recomputes.

In [ ]:
before = await rag.query("MATCH (e:__Entity__) RETURN count(e)")
summary = await rag.finalize()
after = await rag.query("MATCH (e:__Entity__) RETURN count(e)")

print(f"  entities {before[0][0]} -> {after[0][0]}")
print(f"  merged {summary.entities_deduplicated}")
print(f"  names left under more than one label: {summary.unmerged_name_collisions or 'none'}")

In [ ]:
await look(
    "THE POINT: entities reachable from both a PDF and a CSV",
    """MATCH (e:__Entity__)-[:MENTIONED_IN]->(:Chunk)<-[:PART_OF]-(d:Document)
WITH e, collect(DISTINCT d.id) AS sources
WHERE size(sources) > 1
RETURN e.name AS entity, sources
ORDER BY entity""",
    "each of these is ONE node holding both the prose and the numbers. "
    "If this is empty, nothing joined and the rest is theatre",
)

await look(
    "One merged company, seen from both sides",
    """MATCH (o:Organization {name: 'Northwind Energy'})
RETURN o.name, o.org_id, o.employee_count, o.revenue_musd, o.description""",
    "org_id, employee_count and revenue came from the CSV; "
    "description came from the board review",
)

## 7. Questions

Three kinds. Note what this does and does not prove: a correct answer here can
come from retrieval putting both halves in the model's context, which works even
on a disconnected graph. Section 10 does the version that can only work because
the halves actually merged.

In [ ]:
QUESTIONS = [
    ("table only",    "What is the average age of employees at Northwind Energy?"),
    ("document only", "Why did Northwind Energy miss its revenue guidance?"),
    ("both",          "Who works at the company that reported the shortfall, "
                      "and how old are they?"),
    ("both",          "How large is the counterparty named in the market note, "
                      "by headcount and revenue?"),
]

for kind, question in QUESTIONS:
    answer = await rag.completion(question)
    print(f"\n[{kind}] {question}\n  -> {answer.answer}")

## 8. The table changes

A table is a snapshot, not an addition. Re-loading a changed export syncs it:
rows that changed update, rows that appeared arrive, and **rows that disappeared
are removed**. That last one is the case that needs the machinery — a row nobody
rewrites would otherwise sit in the graph forever.

In [ ]:
# employees_v2.csv is the same export a quarter later: Maya is promoted,
# Johan has left, Lene is new. Passing the same document_id says "this is
# the current state of that export", so the load is a re-sync, not an add.
print(open(DATA / "employees_v2.csv").read())

result = await rag.ingest(str(DATA / "employees_v2.csv"), mapping=EMPLOYEES,
                          document_id=EMPLOYEE_EXPORT)
print(f"  {result}")
print(f"  entities_deleted = {result.entities_deleted}  (Johan Berg left the export)")

await look(
    "Who is in the graph now",
    """MATCH (p:Person)
RETURN p.employee_id, p.name, p.title
ORDER BY p.employee_id""",
    "Maya's title is updated, Lene is new, Johan is gone, and E-1 kept its identity",
)

In [ ]:
# Loading it again with nothing changed is a hash comparison, not a rewrite.
result = await rag.ingest(str(DATA / "employees_v2.csv"), mapping=EMPLOYEES,
                          document_id=EMPLOYEE_EXPORT)
print(f"  unchanged re-load -> no_op={result.no_op}")

## 9. Two things the SDK refuses

Both were silent failures once. Neither is now.

In [ ]:
# A table with no mapping would be read as prose: one chunk holding the raw
# commas, and not a single typed column.
try:
    await rag.ingest(str(DATA / "employees.csv"))
except ValueError as exc:
    print("refused:\n")
    print(textwrap.indent(textwrap.fill(str(exc), 74), "  "))

# The escape is real: tickets.csv is prose that happens to live in columns,
# and saying so explicitly still works.
result = await rag.ingest(str(DATA / "tickets.csv"), loader=TextLoader())
print(f"\nallowed with an explicit loader: {result.chunks_indexed} chunk(s)")


### finalize() again, because we just ingested something

That ticket mentions `Northwind Energy`, so reading it as prose created a fresh
entity for a company the graph already had, and `finalize()` ran back in step 6,
before this existed. Left alone the graph would end with two Northwinds: exactly
the split this whole notebook is about.

`finalize()` belongs after the **last** ingest, not after the first batch. It is
cheap to repeat and it is the easiest call to forget.

In [ ]:
nodes = "MATCH (o:Organization) WHERE o.name = 'Northwind Energy' RETURN count(o)"
before = await rag.query(nodes)
summary = await rag.finalize()
after = await rag.query(nodes)
print(f"  Northwind Energy nodes: {before[0][0]} -> {after[0][0]}"
      f"  (merged {summary.entities_deduplicated})")

await look(
    "One node per company, and the ticket now hangs off the same Northwind",
    """MATCH (o:Organization {name: 'Northwind Energy'})-[:MENTIONED_IN]->(:Chunk)
      <-[:PART_OF]-(d:Document)
RETURN DISTINCT d.id AS reached_from
ORDER BY reached_from""",
    "six sources: two PDFs, three CSVs and the ticket, all one company",
)

## 10. Answering by walking the graph

The questions in step 7 came back right, and it is worth being precise about
*why*, because there are two different things people call hybrid and only one of
them needs the graph.

**Both halves in the prompt.** Retrieval fetches the prose chunk and the record
chunks separately, drops them in the model's context, and the model stitches them
together. This works even when the two halves are disconnected islands.

**Walking the join.** One query goes from a sentence in a PDF to a number in a
CSV, because they are on the same node.

Step 7 was mostly the first kind. The generated Cypher for those questions did not
traverse: asked who works at the company that reported the shortfall, it produced
`WHERE o.name CONTAINS 'shortfall'` and returned nothing, and the answer came from
retrieval instead. Correct answer, weaker reason.

So here is the second kind, written by hand, which is the only way to be sure:

In [ ]:
await look(
    "From a sentence in the board review to ages in the HR export, in one walk",
    """MATCH (d:Document)-[:PART_OF]->(c:Chunk)<-[:MENTIONED_IN]-(o:Organization)
      <-[r:RELATES]-(p:Person)
WHERE d.id CONTAINS 'board_review'      // a PDF
  AND c.text CONTAINS 'shortfall'       // the sentence that mentions it
  AND o.org_id = 'ORG-NW'               // a column that only the CSV supplied
  AND r.rel_type = 'WORKS_AT'           // an edge that only the CSV created
RETURN p.name, p.age, p.title
ORDER BY p.name""",
    "Maya Ellison and Tomas Reyes. Every clause in that WHERE comes from a different source, which is only possible because the halves merged",
)

await look(
    "Why it works: the company reached from the PDF carries the CSV's numbers",
    """MATCH (d:Document)-[:PART_OF]->(c:Chunk)<-[:MENTIONED_IN]-(o:Organization)
WHERE d.id CONTAINS 'board_review' AND c.text CONTAINS 'shortfall'
RETURN o.name, o.org_id, o.employee_count, o.revenue_musd
ORDER BY o.name""",
    "reached through prose, answering with columns. Before finalize() these were two nodes and org_id would be null",
)

### The test to apply to any demo of this

Ask whether a question could still be answered if the two halves were never
joined. If it could, the demo is showing retrieval, not a graph. The query above
returns nothing on a disconnected graph, which is what makes it worth running.

## 11. Where to look in the browser

Open **http://localhost:3000**, choose `hybrid_walkthrough`, and try these.

```cypher
// everything, if the graph is small enough to eyeball
MATCH (n)-[r]->(m) RETURN n, r, m LIMIT 200

// the shape of the whole thing: documents, rows, entities
MATCH (d:Document)-[:PART_OF]->(c:Chunk)<-[:MENTIONED_IN]-(e:__Entity__)
RETURN d, c, e LIMIT 60

// only what joined the two halves
MATCH (e:__Entity__)-[:MENTIONED_IN]->(:Chunk)<-[:PART_OF]-(d:Document)
WITH e, collect(DISTINCT d.id) AS sources WHERE size(sources) > 1
RETURN e.name, sources

// a row, verbatim, beside the entity it produced
MATCH (c:Chunk {kind:'record'})<-[:MENTIONED_IN]-(e:__Entity__)
RETURN e.name, c.record_key, c.text LIMIT 20

// aggregate over declared types
MATCH (p:Person)-[r:RELATES]->(o:Organization)
WHERE r.rel_type = 'WORKS_AT'
RETURN o.name, count(p) AS people, avg(p.age) AS mean_age, o.employee_count
```

### What is worth noticing

- A **record Chunk** looks like a paragraph Chunk. That is deliberate: it is why
  provenance, retrieval, update and deletion all work on a row for free.
- The entities extracted from the PDFs are a **cloud of names**, some of them
  incidental. The table's entities are exactly the rows. Both are in one graph.
- `employee_count` is a **number**. Nothing in the prose half could have given
  you that.

In [ ]:
await rag.close()
print("closed. the graph is still there — go and look at it.")